# Playbook · Answer the AI question in a due-diligence questionnaire

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/valuein/valuein/blob/main/examples/playbooks/ddq-ai-question.ipynb)

**Who it is for:** compliance, operations and investor-relations teams answering "how do you stop AI from making up numbers?".

**What you get:** an evidence pack you can attach to an answer: a figure, its filing, an independent check, a test that an invented figure fails, and a CSV for the file.

**Plan needed:** none for the code (free sample, no key). The Workspace section works on the free Benchmark account.

## Step 1 · Name the figure and the question

A due-diligence question usually reads: "When an AI states a financial figure, how do you know it is real?". The answer is a receipt. Start with one figure an AI might quote.

In [1]:
%pip install -q "valuein-sdk>=1.1.0" matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings

import pandas as pd

from valuein_sdk import ValueinClient, compute_fact_id

warnings.filterwarnings("ignore", message="Defaulting to SAMPLE dataset")
client = ValueinClient()
facts = client.read_table("fact", tickers="AAPL")
fy = facts[(facts["standard_concept"] == "TotalRevenue") & (facts["fiscal_period"] == "FY")
           & (facts["period_end"] == "2025-09-27")]
row = fy.sort_values(["accepted_at", "priority"]).iloc[0]  # the figure as first filed
print(f"Apple total revenue, fiscal year ended {row['period_end']:%Y-%m-%d}: {row['numeric_value'] / 1e9:,.3f} (USD billions)")
print("accession:", row["accession_id"], "| accepted:", row["accepted_at"])

Using the free sample means accepting the Valuein Data Licence: valuein.biz/terms#data-licence


Apple total revenue, fiscal year ended 2025-09-27: 416.161 (USD billions)
accession: 0000320193-25-000079 | accepted: 2025-10-31 10:01:26+00:00


### Attach its fact_id

Every figure carries a `fact_id`: a 64-character identifier built from the company, the filing, the concept, the period and the unit. It is the figure's name.

In [3]:
print("fact_id:", row["fact_id"])
print("concept:", row["concept"], "->", row["standard_concept"])

fact_id: 97960097ffc0ea39388b21822abea04be91df63698bec0422f8e4327b0a5247b
concept: us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax -> TotalRevenue


**Check it worked:** revenue of 416.161 (USD billions) from accession 0000320193-25-000079.

Then: a 64-character id and the XBRL concept the filing used, mapped to `TotalRevenue`.

## Step 2 · Check it yourself, without trusting us

Rebuild the `fact_id` locally from the filing facts. No Valuein server is involved, so the check does not depend on our word.

In [4]:
rebuilt = compute_fact_id(row["entity_id"], row["accession_id"], row["concept"],
                          row["period_end"].date(), row["unit"])
print("from the data :", row["fact_id"])
print("rebuilt here  :", rebuilt)
print("match         :", rebuilt == row["fact_id"])

from the data : 97960097ffc0ea39388b21822abea04be91df63698bec0422f8e4327b0a5247b
rebuilt here  : 97960097ffc0ea39388b21822abea04be91df63698bec0422f8e4327b0a5247b
match         : True


### Ask the server to confirm it

An AI agent uses the same check through MCP. This cell calls the public endpoint anonymously.

In [5]:
import json
import os
import urllib.request

MCP_URL = "https://mcp.valuein.biz/mcp"


def mcp_call(tool: str, arguments: dict) -> dict:
    """Call one MCP tool over Streamable HTTP (JSON-RPC) and return its decoded result."""
    headers = {
        "content-type": "application/json",
        "accept": "application/json, text/event-stream",
        "user-agent": "valuein-playbook/1.0",
    }
    if os.environ.get("VALUEIN_API_KEY"):  # optional: your plan's data instead of the free sample
        headers["authorization"] = "Bearer " + os.environ["VALUEIN_API_KEY"]
    body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": "tools/call",
                       "params": {"name": tool, "arguments": arguments}}).encode()
    with urllib.request.urlopen(urllib.request.Request(MCP_URL, body, headers), timeout=60) as resp:
        raw = resp.read().decode()
    if raw.startswith("event:"):  # a server may answer as a server-sent event
        raw = next(line[5:] for line in raw.splitlines() if line.startswith("data:"))
    message = json.loads(raw)
    if "error" in message:
        raise RuntimeError(message["error"].get("message", "MCP error"))
    return json.loads(message["result"]["content"][0]["text"])

In [6]:
reference = f"{row['entity_id']}-{row['fact_id']}"
confirmed = mcp_call("verify_facts", {"facts": [{"fact_id": reference}]})["results"][0]
print("found     :", confirmed["found"])
print("accession :", confirmed["lineage"]["accession_id"])
print("value     :", confirmed["lineage"]["display"])

found     : True
accession : 0000320193-25-000079
value     : $416.16B


**Check it worked:** `match: True`.

Then: `found: True`, the same accession and `$416.16B`.

## Step 3 · Show that an invented figure fails

This is the control a reviewer wants to see: a reference an agent made up is not confirmed, so it cannot pass off a number it invented. The server answers with a refusal code, never with a figure.

In [7]:
invented = reference[:-8] + "00000000"  # looks like a real reference, matches no filing
reply = mcp_call("verify_facts", {"facts": [{"fact_id": invented}]})
entry = (reply.get("results") or [{}])[0]
invented_verified = bool(entry.get("found"))
print("verified:", invented_verified)
print("server says:", entry.get("error", {}).get("code") or reply.get("error", {}).get("code"))

verified: False
server says: ENTITLEMENT_DENIED


### Link the filing

Give the reviewer the document itself, on sec.gov.

In [8]:
links = client.filing_links("AAPL", form_types=["10-K"], limit=5)
hit = links[links["accession_id"] == row["accession_id"]].iloc[0]
print("filing index :", hit["sec_url"])
print("open at figure:", hit["inline_viewer_url"] or hit["document_url"])

filing index : https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/0000320193-25-000079-index.htm
open at figure: https://www.sec.gov/ix?doc=/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm


**Check it worked:** `verified: False` and a server code (a refusal), never a value.

Then: two sec.gov links for accession 0000320193-25-000079.

## Step 4 · Export the evidence

One row for the file: the figure, its id, the independent check and the filing.

In [9]:
evidence = pd.DataFrame([{
    "company": "Apple Inc.", "cik": row["entity_id"], "concept": row["standard_concept"],
    "period_end": f"{row['period_end']:%Y-%m-%d}", "value_usd": row["numeric_value"],
    "fact_id": row["fact_id"], "rebuilt_matches": rebuilt == row["fact_id"],
    "server_found": confirmed["found"], "invented_verified": invented_verified,
    "accession_id": row["accession_id"], "filing_url": hit["sec_url"],
}])
evidence.to_csv("ddq_evidence.csv", index=False)
print("written: ddq_evidence.csv")
client.close()
evidence.T

written: ddq_evidence.csv


,0
company,Apple Inc.
cik,0000320193
concept,TotalRevenue
period_end,2025-09-27
value_usd,416161000000.0
fact_id,97960097ffc0ea39388b21822abea04be91df63698bec0...
rebuilt_matches,True
server_found,True
invented_verified,False
accession_id,0000320193-25-000079


**Check it worked:** a one-row CSV with `rebuilt_matches` True, `server_found` True and `invented_verified` False.

## In the Workspace

The Workspace applies this to a whole report, not one figure.

1. Open Chat and ask for the analysis a reviewer cares about, for example: "Summarise Apple's fiscal 2025 results. Cite the filing for every figure."
2. Save it as a report. Open its Figures panel: every figure is checked against its filing; a figure found in the filing is shown with where it was found, and one the check cannot confirm is sent to a person, never shown as a pass.
3. Sign off the report. The sign-off records the check, so the evidence travels with the document.
4. Attach the report's receipt, with this notebook's CSV, to the questionnaire answer.

More: [`docs/COMPLIANCE_AND_DDQ.md`](../../docs/COMPLIANCE_AND_DDQ.md)

**Next:** [Connect your AI agent](connect-your-agent.ipynb)

_Index names are trademarks of their owners; Valuein is not affiliated with or endorsed by any index provider._